## Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import DataFrame

## File Path Define

In [0]:
orders_path = "/Volumes/data_engineering_2026/data_quality/data_quality_sources/olist_orders_dataset.csv"

customers_path = "/Volumes/data_engineering_2026/data_quality/data_quality_sources/olist_customers_dataset.csv"

items_path = "/Volumes/data_engineering_2026/data_quality/data_quality_sources/olist_order_items_dataset.csv"


print("File apaths defined successfully")
print(f"orders_path: {orders_path}")
print(f"customers_path: {customers_path}")
print(f"items_path: {items_path}")

## Loaded Raw orders Csv

In [0]:
orders_schema = StructType([
	StructField("order_id", StringType(), True),
	StructField("customer_id", StringType(), True),
	StructField("order_status", StringType(), True),
	StructField("order_pruchase_timestamp", TimestampType(), True),
	StructField("order_approved_at", TimestampType(), True),
	StructField("order_delivered_carrier_date", TimestampType(), True),
	StructField("order_order_delivered_customer_date", TimestampType(), True),
	StructField("order_estimated_delivery_date", TimestampType(), True)])

orders = spark.read\
	.format("csv")\
	.option("header", "true")\
	.schema(orders_schema)\
	.option("mode", "failfast")\
	.load(orders_path)

print(f'orders loaded successfully with {orders.count()} rows')
print(f'Number of Columns : {len(orders.columns)}')
orders.columns

## Loaded Raw Customers Csv


In [0]:
customers_schema = StructType([
	StructField("customer_id", StringType(), True),
	StructField("customer_unique_id", StringType(), True),
	StructField("customer_zip_code_prefix", IntegerType(), True),
	StructField("customer_city", StringType(), True),
	StructField("customer_state", StringType(), True),])

customers = spark.read\
	.format("csv")\
	.option("header", "true")\
	.schema(customers_schema)\
	.option("mode", "failfast")\
	.load(customers_path)

print(f'orders loaded successfully with {customers.count()} rows')
print(f'Number of Columns : {len(customers.columns)}')
customers.columns

## Loaded the Raw Order list Csv

In [0]:
orderslist_schema = StructType([
	StructField("order_id", StringType(), True),
	StructField("order_item_id", IntegerType(), True),
	StructField("product_id", StringType(), True),
	StructField("seller_id", StringType(), True),
	StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DecimalType(10,2), True),
    StructField("freight_value", DecimalType(10,2), True),])

orderslists = spark.read\
	.format("csv")\
	.option("header", "true")\
	.schema(orderslist_schema)\
	.option("mode", "failfast")\
	.load(items_path)

print(f'orders loaded successfully with {orderslists.count()} rows')
print(f'Number of Columns : {len(orderslists.columns)}')
orderslists.columns

## Write all the csv

In [0]:
orders.write\
    .mode("append")\
    .saveAsTable("data_engineering_2026.bronze.orders")

customers.write\
    .mode("append")\
    .saveAsTable("data_engineering_2026.bronze.customers")

orderslists.write\
    .mode("append")\
    .saveAsTable("data_engineering_2026.bronze.items")

## Verification

In [0]:
bronze_orders = spark.read\
  .format("delta")\
  .table("data_engineering_2026.bronze.orders")

bronze_customers = spark.read\
    .format("delta")\
    .table("data_engineering_2026.bronze.customers")

bronze_items = spark.read\
    .format("delta")\
    .table("data_engineering_2026.bronze.items")


print(f"  Orders:{bronze_orders.count():>12,} records") 
print(f"  customers:{bronze_customers.count():>12,} records") 
print(f"  items:{bronze_items.count():>12,} records") 